# Step9 (Socavamiento) — Análisis Espacial (Ecuador, 2015–2025)

**Proyecto:** HCI Ecuador (Hazard Chronicity Index)
**Fase:** 9 — Hotspots (Getis-Ord Gi*), autocorrelación local (Moran's I),
densidad kernel (KDE) y agregación hexagonal (Fase 0 §8) — **versión
independiente para el peligro Socavamiento**, primera de 4 tras la decisión
de Jerson (2026-08-17) de separar completamente el análisis por tipo de
amenaza en vez de usar solo la vista combinada `"TODOS"`.
**Autor:** Jerson Achicanoy (FAO OER) · implementación asistida
**Fecha:** 2026-08-17

**Relación con `Step9_Ecuador_Spatial_Analysis.ipynb` (el original, vista
`TODOS`):** mismo método exacto (Queen+KNN, Gi*/Local Moran's I sobre las
5 capas, KDE ponderado por severidad, hexágonos H3 res5) — la única
diferencia es la fuente de datos: aquí se filtra `hci_composite_canton`
y `hazard_event_geo` a `hazard_type == "scour"` antes de correr el
análisis, en vez de usar la vista `"TODOS"`. El notebook original NO se
modifica ni se reemplaza — sigue siendo el análisis nacional combinado,
útil para una lectura agregada de cronicidad. Este notebook responde una
pregunta distinta: ¿dónde se agrupa específicamente la cronicidad de
**socavamiento**, sin que la mezcle con inundación/deslizamientos/aluvión?

**Decisión confirmada por Jerson antes de programar (AskUserQuestion,
2026-08-17):** 67 de los 221 cantones nunca registraron un evento de
socavamiento — se les asigna HCI e índices = 0 (cronicidad de socavamiento
genuinamente nula, no un dato faltante) para mantener el universo completo
de 221 cantones y el mismo grafo de vecindad Queen+KNN ya validado en el
notebook original — la topología de vecindad depende solo de la geometría
administrativa, no del valor de las variables, así que es idéntica en
ambos notebooks.

**Decisiones heredadas del notebook original (2026-08-06, sin cambios):**
- **Variables:** Gi* y Local Moran's I sobre las 5 capas — HCI + los 4
  sub-índices (Severidad, Recurrencia, Persistencia, Magnitud ENSO).
- **Pesos espaciales:** contigüidad Queen orden 1 + respaldo KNN=1 para
  islas (ver Sección 2).
- **KDE:** eventos de socavamiento ponderados por su score de severidad
  individual.
- **Hexágonos:** resolución H3 5 (~9.85 km de borde).

**No modifica** ninguna salida de las Fases 1-8 ni del `Step9` original —
exporta a archivos nuevos con sufijo `_scour`.

**English:** Step9 — Análisis Espacial (Ecuador, 2015–2025)

**Proyecto:** HCI Ecuador (Hazard Chronicity Index)
**Fase:** 9 — Hotspots (Getis-Ord Gi*), autocorrelación local (Moran's I),
densidad kernel (KDE) y agregación hexagonal (Fase 0 §8)
**Autor:** Jerson Achicanoy (FAO OER) · implementación asistida
**Fecha:** 2026-08-06

**Objetivo de este notebook:** identificar patrones de agrupamiento espacial
en el HCI y sus 4 sub-índices (¿hay clusters de cantones crónicos, o el
patrón es aleatorio?), producir una superficie continua de densidad de
impacto (no limitada a fronteras administrativas), y una vista agregada en
hexágonos para comparar unidades de tamaño uniforme (los cantones de
Ecuador varían mucho en área, lo que puede sesgar la lectura visual de un
mapa coroplético).

**Decisiones confirmadas por Jerson antes de programar (AskUserQuestion,
2026-08-06) — ver justificación completa en
`docs/methodology/09_spatial_analysis.md`:**
- **Variables:** Gi* y Local Moran's I se corren sobre las 5 capas — HCI
  compuesto + los 4 sub-índices (Severidad, Recurrencia, Persistencia,
  Magnitud ENSO) — no solo el HCI final, para poder diagnosticar si un
  hotspot de HCI está dominado por un componente específico.
- **Pesos espaciales:** delegado explícitamente a criterio técnico
  ("el mejor para este tipo de ejercicios"). Se usa **contigüidad Queen**
  de orden 1 (estándar en econometría espacial para polígonos
  administrativos) con **respaldo KNN=1** para unidades sin vecino por
  contigüidad — resuelto en la Sección 1, con verificación de que el
  respaldo conecta lo que geográficamente corresponde (Galápagos entre
  sí, no con el continente).
- **KDE:** eventos ponderados por su score de severidad individual (no
  densidad simple de conteo) — una zona con pocos eventos graves debe
  resaltar más que una con muchos eventos leves.
- **Hexágonos:** resolución H3 5 (~9.85 km de borde promedio, la más
  cercana a los "~10 km" pedidos).

**No modifica** ninguna salida de las Fases 1-8 — trabaja únicamente sobre
sus resultados ya validados.

**English:** Spatial Analysis

Identifies where chronicity clusters: Getis-Ord Gi* hotspots, Local Moran's I clusters and outliers, severity-weighted kernel density and hexagonal (H3) aggregation, using the composite index and the geolocated events.

Outputs: `hotspots_*`, `local_moran_*`, `kde_*` and `hexagonos_*` files in `data/processed/`.

## 0. Configuración del entorno

**English:** 0. Configuración del entorno

**English:** 0. Setup

In [ ]:
# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent  # notebook vive en <repo>/notebooks/
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helper to create output folders
from src.utilidades.io_utils import ensure_dir

# Numerical operations
import numpy as np
# Tabular data handling
import pandas as pd
# Geospatial data handling
import geopandas as gpd
# Queen contiguity weights and the generic weights class
from libpysal.weights import Queen, W
# Getis-Ord Gi* and Local Moran's I statistics
from esda import G_Local, Moran_Local
# H3 hexagonal grid library
import h3
# Gaussian kernel density estimator
from scipy.stats import gaussian_kde

# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)
# Create a logger that writes to the logs folder
logger = get_logger("step9_spatial_analysis", config, log_dir=PROJECT_ROOT / "logs")
logger.info("Fase 9 iniciada — configuración cargada desde %s", CONFIG_PATH)

# Folder with processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())
# Spatial analysis settings from the config
SPA_CFG = config["spatial_analysis"]
# Variables to analyze (composite index and its components)
VARIABLES = SPA_CFG["variables"]
# Number of permutations for significance tests
PERMUTATIONS = SPA_CFG["significance"]["permutations"]
# P-value thresholds for 99%, 95% and 90% confidence
P_THRESHOLDS = SPA_CFG["significance"]["p_thresholds"]  # [0.01, 0.05, 0.10]
# H3 resolution of the hexagons
H3_RES = SPA_CFG["hexagons"]["h3_resolution"]
# Kernel bandwidth method
KDE_BW = SPA_CFG["kde"]["bw_method"]
# Kernel grid resolution in kilometres
KDE_GRID_KM = SPA_CFG["kde"]["grid_resolution_km"]
# Metric coordinate system used for distances
CRS_METRIC = SPA_CFG["kde"]["crs_metric"]

# Fixed random seed for reproducibility
RNG_SEED = 42  # fijo, para que los p-valores de permutación sean reproducibles

logger.info("Fase 9 iniciada. Variables a analizar: %s", VARIABLES)

## 1. Insumos: geometría de cantones + HCI compuesto (scour) + eventos de socavamiento

Se reutilizan las mismas salidas ya validadas de fases anteriores, sin
recalcularlas:
- `adm2_canton.geojson` (Fase 2): 221 polígonos de cantón.
- `hci_composite_canton.csv` (Fase 8, corregido): HCI + 4 sub-índices,
  filtrado a `hazard_type == "scour"` (214/221 cantones con historial de
  socavamiento) — los 67 cantones restantes se rellenan con 0, ver decisión
  confirmada en la introducción de este notebook.
- `hazard_event_geo.gpkg` (Fase 1/2, corregido): filtrado a
  `hazard_type == "scour"` — insumo del KDE y la agregación hexagonal.

**English:** 1. Insumos: geometría de cantones + HCI compuesto + eventos geolocalizados

Se reutilizan tres salidas ya validadas de fases anteriores, sin
recalcularlas:
- `adm2_canton.geojson` (Fase 2): 221 polígonos de cantón.
- `hci_composite_canton.csv` (Fase 8, corregido): HCI + 4 sub-índices,
  vista `"TODOS"` (todos los tipos de peligro combinados) — la vista
  relevante para hotspots nacionales de cronicidad.
- `hazard_event_geo.gpkg` (Fase 1/2, corregido): 21,743 eventos
  puntuales, insumo del KDE y la agregación hexagonal.

**English:** 1. Inputs: canton geometry, composite index and geolocated events

In [ ]:
# --- Geometría de cantones ---
# Read canton polygons
gdf_canton = gpd.read_file(processed_dir / "adm2_canton.geojson")
# Make canton codes strings
gdf_canton["canton_code"] = gdf_canton["canton_code"].astype(str)
# Log the number of polygons and the CRS
logger.info("Cantones cargados: %d polígonos, CRS=%s", len(gdf_canton), gdf_canton.crs)

# --- HCI compuesto, filtrado a hazard_type == "scour" ---
# Hazard type analysed in this notebook
HAZARD_TYPE = "scour"
# --- HCI compuesto (vista TODOS únicamente — la relevante para hotspots nacionales) ---
# Read the composite index at canton level
hci = pd.read_csv(processed_dir / "hci_composite_canton.csv", dtype={"canton_code": str})
# Keep only the rows of this hazard type
hci_scour = hci[hci["hazard_type"] == HAZARD_TYPE].copy()
# Number of cantons with at least one event of this hazard
n_con_historial = len(hci_scour)
logger.info(
    "HCI (hazard_type=scour) cargado: %d/221 cantones con historial de socavamiento",
    n_con_historial,
)

# Rellena con 0 los cantones sin ningún evento de socavamiento histórico —
# decisión confirmada por el equipo (2026-08-17): cronicidad genuinamente
# nula, no un dato faltante. Mantiene el universo completo de 221 cantones
# y reutiliza exactamente el mismo grafo de vecindad Queen+KNN.
# Full universe of 221 cantons
universo_cantones = gdf_canton[["canton_code"]].drop_duplicates()
# Left-join so cantons without history stay in the table
hci_scour_completo = universo_cantones.merge(hci_scour, on="canton_code", how="left")
# Count cantons that will be filled with zero
n_rellenados = hci_scour_completo[VARIABLES].isna().any(axis=1).sum()
# Fill missing index values with 0 (no events = no chronicity, not missing data)
hci_scour_completo[VARIABLES] = hci_scour_completo[VARIABLES].fillna(0.0)
# Missing event counts become 0
hci_scour_completo["n_eventos"] = hci_scour_completo["n_eventos"].fillna(0).astype(int)
# Check the full 221-canton universe is kept
assert len(hci_scour_completo) == 221, f"se esperaban 221 cantones tras el relleno, hay {len(hci_scour_completo)}"
# Check the filled-canton count matches the expectation
assert n_rellenados == 221 - n_con_historial, "el conteo de cantones rellenados con 0 no coincide con lo esperado"
logger.info(
    "Relleno con HCI=0 aplicado a %d cantones sin historial de socavamiento: universo completo = 221",
    n_rellenados,
)

# Une geometría + HCI (scour, universo completo) por canton_code
# Une geometría + HCI por canton_code (llave estable desde Fase 1)
# Join the index values to the canton polygons
gdf = gdf_canton.merge(
    hci_scour_completo[["canton_code"] + VARIABLES],
    on="canton_code", how="left",
)
# Check no canton lacks a value
assert gdf[VARIABLES].isna().sum().sum() == 0, "hay cantones sin valor de HCI o algún sub-índice — revisar el merge"
# Reset the index so positions match the weights matrix
gdf = gdf.reset_index(drop=True)  # índice posicional limpio, requerido por libpysal
logger.info("Tabla espacial de trabajo (scour): %d cantones x %d variables", len(gdf), len(VARIABLES))

# --- Eventos geolocalizados, filtrados a socavamiento (para KDE y hexágonos) ---
# Read all georeferenced events
he_todos = gpd.read_file(processed_dir / "hazard_event_geo.gpkg", layer="hazard_event")
# Keep only events of this hazard type (used for KDE and hexagons)
he = he_todos[he_todos["hazard_type"] == HAZARD_TYPE].copy()
logger.info("Eventos de socavamiento cargados: %d de %d totales", len(he), len(he_todos))

## 2. Matriz de pesos espaciales: contigüidad Queen + respaldo KNN

**Por qué Queen y no Rook:** contigüidad Queen cuenta como vecinos a los
polígonos que comparten frontera *o* un solo vértice — más permisiva que
Rook (solo frontera compartida), y la convención por defecto en la
literatura de econometría espacial (Anselin) para polígonos
administrativos irregulares como cantones.

**El problema de las islas:** una matriz de contigüidad pura falla si
algún polígono no toca a ningún otro (queda "aislado", sin vecinos,
lo cual rompe el cálculo de Gi*/Local Moran's I para esa unidad). Se
verifica explícitamente cuántos cantones caen en este caso antes de
decidir cómo resolverlo — no se asume.

**English:** 2. Matriz de pesos espaciales: contigüidad Queen + respaldo KNN

**Por qué Queen y no Rook:** contigüidad Queen cuenta como vecinos a los
polígonos que comparten frontera *o* un solo vértice — más permisiva que
Rook (solo frontera compartida), y la convención por defecto en la
literatura de econometría espacial (Anselin) para polígonos
administrativos irregulares como cantones.

**El problema de las islas:** una matriz de contigüidad pura falla si
algún polígono no toca a ningún otro (queda "aislado", sin vecinos,
lo cual rompe el cálculo de Gi*/Local Moran's I para esa unidad). Se
verifica explícitamente cuántos cantones caen en este caso antes de
decidir cómo resolverlo — no se asume.

**English:** 2. Spatial weights: Queen contiguity with KNN fallback

In [ ]:
# Queen contiguity: cantons sharing a border or a corner are neighbors
w_queen = Queen.from_dataframe(gdf, use_index=True)
# Log the coverage check
logger.info(
    "Queen: %d vecindades, %d cantones sin vecino (islas): %s",
    w_queen.n, len(w_queen.islands), w_queen.islands,
)

# Identifica qué cantones son (para que quede documentado, no solo el índice)
# Check for islands (cantons without neighbors)
if w_queen.islands:
    # Names of the island cantons
    islas_nombres = gdf.loc[w_queen.islands, "canton_name_official"].tolist()
    # Log the island cantons
    logger.info("Cantones sin vecino por contigüidad: %s", islas_nombres)

In [ ]:
# --- Respaldo KNN=1 SOLO para las islas detectadas ---
# Se calcula sobre centroides en un CRS métrico (no en grados lat/lon, donde
# la distancia no es comparable en todas las direcciones).
# --- Respaldo KNN=1 SOLO para las islas detectadas ---
# Se calcula sobre centroides en un CRS métrico (no en grados lat/lon, donde
# la distancia no es comparable en todas las direcciones).
# Reproject to a metric CRS to measure distances
gdf_m = gdf.to_crs(CRS_METRIC)
# Canton centroids
centroids = gdf_m.geometry.centroid

# Neighbor lists from the Queen weights
neighbors = {i: list(w_queen.neighbors[i]) for i in range(len(gdf))}
# Binary weights for each neighbor
weights_bin = {i: [1.0] * len(neighbors[i]) for i in neighbors}

# Loop over the island cantons
for idx in w_queen.islands:
    # Distances from the island to every other centroid
    dists = centroids.distance(centroids.iloc[idx])
    # Sort by distance, excluding the island itself
    dists = dists.drop(index=idx).sort_values()
    # Index of the nearest canton
    vecino_idx = int(dists.index[0])
    # Distance to the nearest canton in kilometres
    dist_km = dists.iloc[0] / 1000
    # Log a warning so the correction is auditable
    logger.warning(
        "Respaldo KNN aplicado: '%s' no tiene vecino por contigüidad -> "
        "conectado a '%s' (vecino más cercano, %.1f km)",
        gdf.loc[idx, "canton_name_official"], gdf.loc[vecino_idx, "canton_name_official"], dist_km,
    )
    # Conexión simétrica: si A es vecino de B, B también es vecino de A —
    # requerido para que la matriz de pesos sea válida (W debe ser, en la
    # práctica, tratable como no dirigida para Gi*/Local Moran's I).
    # Conexión simétrica: si A es vecino de B, B también es vecino de A —
    # requerido para que la matriz de pesos sea válida (W debe ser, en la
    # práctica, tratable como no dirigida para Gi*/Local Moran's I).
    # Add the nearest canton as a neighbor if missing
    if vecino_idx not in neighbors[idx]:
        # Add the neighbor
        neighbors[idx].append(vecino_idx)
        # Give it a binary weight
        weights_bin[idx].append(1.0)
    # Make the link symmetric
    if idx not in neighbors[vecino_idx]:
        # Add the island as a neighbor of the nearest canton
        neighbors[vecino_idx].append(idx)
        # Give it a binary weight
        weights_bin[vecino_idx].append(1.0)

# Combined weights matrix (Queen plus nearest neighbor)
w_combined = W(neighbors, weights_bin)
# Use binary weights
w_combined.transform = "b"  # binario — para Gi* (Sección 3)
# Check no islands remain
assert len(w_combined.islands) == 0, "quedaron islas sin resolver tras el respaldo KNN"
# Log the size of the combined weights
logger.info("Matriz de pesos combinada (Queen+KNN): 0 islas, %d cantones conectados", w_combined.n)

# Copia row-standardizada para Local Moran's I (Sección 4) — misma topología,
# transformación distinta (convención estándar: Gi* usa binario, Local
# Moran's I usa fila-estandarizado).
# Copia row-standardizada para Local Moran's I (Sección 4) — misma topología,
# transformación distinta (convención estándar: Gi* usa binario, Local
# Moran's I usa fila-estandarizado).
# Row-standardized copy of the weights for Local Moran's I
w_rowstd = W(neighbors, weights_bin)
# Use row-standardized weights
w_rowstd.transform = "r"

**Verificación de que el respaldo KNN conecta lo que corresponde
geográficamente**, no un artefacto: los 3 cantones sin vecino por
contigüidad son, verificado en la Sección 1 previa a programar esto,
San Cristóbal, Isabela y Santa Cruz — los 3 cantones de la provincia de
Galápagos, islas reales en el océano Pacífico. Sus centroides están a
85-94 km entre sí, y a más de 900 km del cantón continental más cercano
— el respaldo KNN=1 los conecta entre ellos, no con el continente, que es
exactamente el resultado geográficamente correcto.

**English:** Check that the KNN fallback connects what it should geographically, not an artefact: the 3 cantons without a contiguity neighbour are San Cristóbal, Isabela and Santa Cruz, the three cantons of Galápagos province, real islands in the Pacific Ocean. Their centroids are 85–94 km apart and more than 900 km from the nearest mainland canton, so the KNN=1 fallback links them to each other and not to the mainland, which is the geographically correct result.

In [ ]:
# Confirma explícitamente en el notebook, no solo en el log de texto arriba
# Loop over the island cantons
for idx in w_queen.islands:
    # Names of the neighbors assigned to the island
    vecinos_nombres = [gdf.loc[j, "canton_name_official"] for j in neighbors[idx]]
    # Print each island with its neighbors
    print(f"{gdf.loc[idx, 'canton_name_official']:25s} -> conectado a: {vecinos_nombres}")

## 3. Getis-Ord Gi* — hotspots/coldspots de cronicidad

Gi* mide si el valor de una unidad *y sus vecinos* es sistemáticamente
alto (hotspot) o bajo (coldspot) respecto al promedio nacional — a
diferencia de un mapa coroplético simple, que solo muestra el valor de
cada cantón de forma aislada. Significancia estadística vía permutación
condicional (999 permutaciones, semilla fija para reproducibilidad) en
vez de asumir normalidad — más robusto para datos espaciales, que casi
nunca son normales.

Clasificación en 6 niveles según el signo de Gi* (z-score) y el
p-valor de permutación: Punto Caliente 99/95/90%, No significativo,
Punto Frío 90/95/99% — la convención de "Hot Spot Analysis" estándar
(ArcGIS/GeoDa).

**English:** 3. Getis-Ord Gi* — hotspots/coldspots de cronicidad

Gi* mide si el valor de una unidad *y sus vecinos* es sistemáticamente
alto (hotspot) o bajo (coldspot) respecto al promedio nacional — a
diferencia de un mapa coroplético simple, que solo muestra el valor de
cada cantón de forma aislada. Significancia estadística vía permutación
condicional (999 permutaciones, semilla fija para reproducibilidad) en
vez de asumir normalidad — más robusto para datos espaciales, que casi
nunca son normales.

Clasificación en 6 niveles según el signo de Gi* (z-score) y el
p-valor de permutación: Punto Caliente 99/95/90%, No significativo,
Punto Frío 90/95/99% — la convención de "Hot Spot Analysis" estándar
(ArcGIS/GeoDa).

**English:** 3. Getis-Ord Gi*: chronicity hotspots and coldspots

In [ ]:
# Classify a Gi* result as hotspot or coldspot at 99%, 95% or 90%
def clasificar_gi(z, p):
    """Clasifica un resultado Gi* en 7 niveles según signo y significancia.

    Args:
        z: z-score de Gi*.
        p: p-valor de permutación (p_sim).

    Returns:
        Etiqueta de clasificación (str).
    """
    # Not significant above the 90% threshold
    if p > P_THRESHOLDS[2]:  # > 0.10
        # Label for non-significant cantons
        return "No significativo"
    # Confidence level of the result
    nivel = "99%" if p <= P_THRESHOLDS[0] else ("95%" if p <= P_THRESHOLDS[1] else "90%")
    # Hotspot when the z-score is positive, coldspot otherwise
    return f"Punto Caliente {nivel}" if z > 0 else f"Punto Frío {nivel}"


# Dictionary that collects the Gi* results
gi_resultados = {"canton_code": gdf["canton_code"], "canton_name_official": gdf["canton_name_official"]}

# Loop over the variables
for var in VARIABLES:
    # Values of the variable
    y = gdf[var].values
    # Getis-Ord Gi* with permutation-based significance
    gi = G_Local(y, w_combined, transform="B", star=True, permutations=PERMUTATIONS, seed=RNG_SEED)
    # Store the z-scores
    gi_resultados[f"gi_z_{var}"] = gi.Zs
    # Store the pseudo p-values
    gi_resultados[f"gi_p_{var}"] = gi.p_sim
    # Store the hotspot or coldspot class
    gi_resultados[f"gi_clase_{var}"] = [clasificar_gi(z, p) for z, p in zip(gi.Zs, gi.p_sim)]
    # Count hotspots
    n_hot = sum(1 for c in gi_resultados[f"gi_clase_{var}"] if "Caliente" in c)
    # Count coldspots
    n_cold = sum(1 for c in gi_resultados[f"gi_clase_{var}"] if "Frío" in c)
    # Log the counts
    logger.info("Gi* (%s): %d puntos calientes, %d puntos fríos, de %d cantones", var, n_hot, n_cold, len(y))

# Build the Gi* table
gi_df = pd.DataFrame(gi_resultados)
# Preview the table
gi_df.head()

## 4. Local Moran's I — clusters y valores atípicos espaciales

**English:** 4. Local Moran's I — clusters y valores atípicos espaciales

**English:** 4. Local Moran's I: clusters and spatial outliers

In [ ]:
# Local Moran's I clasifica cada unidad en 4 cuadrantes (HH/LH/LL/HL) según
# su propio valor y el promedio de sus vecinos:
# HH = cantón alto rodeado de vecinos altos (cluster de cronicidad)
# LL = cantón bajo rodeado de vecinos bajos (cluster de baja cronicidad)
# HL / LH = valores atípicos espaciales (un cantón que no se parece a su entorno)
# Local Moran's I clasifica cada unidad en 4 cuadrantes (HH/LH/LL/HL) según
# su propio valor y el promedio de sus vecinos:
# HH = cantón alto rodeado de vecinos altos (cluster de cronicidad)
# LL = cantón bajo rodeado de vecinos bajos (cluster de baja cronicidad)
# HL / LH = valores atípicos espaciales (un cantón que no se parece a su entorno)
# Labels of the four Moran quadrants
QUADRANT_LABELS = {1: "Alto-Alto (HH)", 2: "Bajo-Alto (LH)", 3: "Bajo-Bajo (LL)", 4: "Alto-Bajo (HL)"}

# Dictionary that collects the Local Moran results
moran_resultados = {"canton_code": gdf["canton_code"], "canton_name_official": gdf["canton_name_official"]}

# Loop over the variables
for var in VARIABLES:
    # Values of the variable
    y = gdf[var].values
    # Local Moran's I with permutation-based significance
    ml = Moran_Local(y, w_rowstd, permutations=PERMUTATIONS, seed=RNG_SEED)
    # Mark significant cantons at the 95% threshold
    sig = ml.p_sim <= P_THRESHOLDS[1]  # significativo al 95%
    # Store the local statistic
    moran_resultados[f"moran_i_{var}"] = ml.Is
    # Store the pseudo p-values
    moran_resultados[f"moran_p_{var}"] = ml.p_sim
    # Store the quadrant label for significant cantons
    moran_resultados[f"moran_cuadrante_{var}"] = [
        QUADRANT_LABELS[q] if s else "No significativo" for q, s in zip(ml.q, sig)
    ]
    # Count high-high clusters
    n_hh = sum(1 for c in moran_resultados[f"moran_cuadrante_{var}"] if "HH" in c)
    # Count low-low clusters
    n_ll = sum(1 for c in moran_resultados[f"moran_cuadrante_{var}"] if "LL" in c)
    # Log the cluster counts
    logger.info("Local Moran's I (%s): %d clusters HH, %d clusters LL (95%%), de %d cantones", var, n_hh, n_ll, len(y))

# Build the Local Moran table
moran_df = pd.DataFrame(moran_resultados)
# Preview the table
moran_df.head()

## 5. Validación de las secciones 3-4

Antes de exportar: ningún valor nulo/infinito, cobertura completa de
221 cantones en ambas tablas (incluidos los 67 rellenados con 0), y una
verificación de sentido común — Quito, que lidera el HCI combinado
(Fase 8 §7) y también el HCI-scour específico, debería aparecer como
parte de un hotspot o al menos no como punto frío en la capa `hci`.

**English:** 5. Validación de las secciones 3-4

Antes de exportar: ningún valor nulo/infinito, cobertura completa de
221 cantones en ambas tablas, y una verificación de sentido común —
Quito (el cantón #1 en HCI, Fase 8 §7) debería aparecer como parte de
un hotspot o al menos no como punto frío en la capa `hci`.

**English:** 5. Validation of sections 3-4

In [ ]:
# Loop over both result tables
for df_check, nombre in [(gi_df, "Gi*"), (moran_df, "Local Moran's I")]:
    # Check all 221 cantons are present
    assert len(df_check) == 221, f"{nombre}: se esperaban 221 cantones, hay {len(df_check)}"
    # Numeric result columns
    cols_numericas = [c for c in df_check.columns if c.startswith(("gi_z_", "gi_p_", "moran_i_", "moran_p_"))]
    # Check there are no missing values
    assert df_check[cols_numericas].isna().sum().sum() == 0, f"{nombre}: hay valores nulos"
    # Check all values are finite
    assert np.isfinite(df_check[cols_numericas].values).all(), f"{nombre}: hay valores infinitos"

# Gi* class of Quito as a sanity check
quito_gi = gi_df.loc[gi_df["canton_name_official"].str.contains("Quito", na=False), "gi_clase_hci"].values
# Local Moran class of Quito as a sanity check
quito_moran = moran_df.loc[moran_df["canton_name_official"].str.contains("Quito", na=False), "moran_cuadrante_hci"].values
# Log the Quito result
logger.info("Verificación de sentido común — Quito: Gi*(hci)=%s, Moran(hci)=%s", quito_gi, quito_moran)
# Print the Gi* result for Quito
print("Gi* Quito (hci):", quito_gi)
# Print the Local Moran result for Quito
print("Moran's I Quito (hci):", quito_moran)

## 6. Densidad Kernel (KDE) ponderada por severidad de evento

A diferencia de las secciones 3-4 (que trabajan con valores ya agregados
por cantón), el KDE trabaja directamente sobre los 21,743 puntos de
evento — produce una superficie continua, sin las fronteras artificiales
de los polígonos administrativos, útil para identificar corredores o
zonas de impacto que cruzan límites cantonales.

**Ponderación:** se recalcula un score de severidad a nivel evento
(misma fórmula de pilares de Fase 4 §4.1: población/vivienda/
agricultura/ganadería/infraestructura, con los mismos multiplicadores de
`config.yaml`), normalizado 0-1 a nivel nacional — no es el mismo
`severity_index` de Fase 4 (que agrega por máximo histórico y normaliza
por cantón), es una versión a nivel evento individual, calculada aquí
específicamente para ponderar el KDE. Un evento con severidad 0 sigue
contribuyendo un mínimo de densidad (no se excluye), pero pesa mucho
menos que un evento severo.

**English:** 6. Densidad Kernel (KDE) ponderada por severidad de evento

A diferencia de las secciones 3-4 (que trabajan con valores ya agregados
por cantón), el KDE trabaja directamente sobre los 21,743 puntos de
evento — produce una superficie continua, sin las fronteras artificiales
de los polígonos administrativos, útil para identificar corredores o
zonas de impacto que cruzan límites cantonales.

**Ponderación:** se recalcula un score de severidad a nivel evento
(misma fórmula de pilares de Fase 4 §4.1: población/vivienda/
agricultura/ganadería/infraestructura, con los mismos multiplicadores de
`config.yaml`), normalizado 0-1 a nivel nacional — no es el mismo
`severity_index` de Fase 4 (que agrega por máximo histórico y normaliza
por cantón), es una versión a nivel evento individual, calculada aquí
específicamente para ponderar el KDE. Un evento con severidad 0 sigue
contribuyendo un mínimo de densidad (no se excluye), pero pesa mucho
menos que un evento severo.

**English:** 6. Kernel density weighted by event severity

In [ ]:
# Severity index settings from the config
SEV_CFG = config["severity_index"]
# Weight multiplier for destroyed assets versus affected ones
W_DESTROYED = SEV_CFG["destroyed_weight_multiplier"]
# Weight multiplier for deaths and missing persons
W_DEATH = SEV_CFG["death_weight_multiplier"]
# Weight multiplier for injured persons
W_INJURED = SEV_CFG["injured_weight_multiplier"]
# Pillar weights from the severity configuration
PESOS = SEV_CFG["pillar_weights"]

# Population score: affected, evacuated and sheltered people plus weighted deaths, missing and injured
he["score_poblacion"] = (
    he["affected_direct"] + he["affected_indirect"] + he["evacuated"] + he["sheltered"]
    + W_DEATH * (he["deaths"] + he["missing_persons"])
    + W_INJURED * he["injured"]
)
# Housing score: affected houses plus weighted destroyed houses
he["score_vivienda"] = he["houses_affected"] + W_DESTROYED * he["houses_destroyed"]
# Agriculture score: affected hectares plus weighted destroyed hectares
he["score_agricultura"] = he["crops_affected_ha"] + W_DESTROYED * he["crops_destroyed_ha"]
# Livestock score: affected animals plus weighted deaths
he["score_ganaderia"] = he["livestock_affected"] + W_DESTROYED * he["livestock_deaths"]
# Infrastructure score: affected and destroyed schools and health facilities
he["score_infraestructura"] = (
    he["schools_affected"] + W_DESTROYED * he["schools_destroyed"]
    + he["health_facilities_affected"] + W_DESTROYED * he["health_facilities_destroyed"]
)

# Normaliza cada pilar 0-1 a nivel nacional (evento a evento, no por cantón)
# antes de combinarlos con los mismos pesos de Fase 4 — así el score final
# queda en una escala comparable [0, 1] utilizable como peso de KDE.
# Normaliza cada pilar 0-1 a nivel nacional (evento a evento, no por cantón)
# antes de combinarlos con los mismos pesos de Fase 4 — así el score final
# queda en una escala comparable [0, 1] utilizable como peso de KDE.
# Names of the pillar score columns
pilares = ["score_poblacion", "score_vivienda", "score_agricultura", "score_ganaderia", "score_infraestructura"]
# Loop over the pillars
for col in pilares:
    # Minimum and maximum of the pillar score
    minimo, maximo = he[col].min(), he[col].max()
    # Range of the pillar score
    rango = maximo - minimo
    # Min-max normalization of the pillar
    he[col.replace("score_", "norm_")] = (he[col] - minimo) / rango if rango > 0 else 0.0

# Event severity score: weighted sum of the normalized pillars
he["event_severity_score"] = (
    PESOS["poblacion"] * he["norm_poblacion"]
    + PESOS["vivienda"] * he["norm_vivienda"]
    + PESOS["agricultura"] * he["norm_agricultura"]
    + PESOS["ganaderia"] * he["norm_ganaderia"]
    + PESOS["infraestructura"] * he["norm_infraestructura"]
)
# Check the event severity score is between 0 and 1
assert he["event_severity_score"].between(0, 1).all(), "event_severity_score fuera de [0, 1]"
# Log the coverage check
logger.info(
    "event_severity_score calculado: media=%.4f, máximo=%.4f (evento %s)",
    he["event_severity_score"].mean(), he["event_severity_score"].max(),
    he.loc[he["event_severity_score"].idxmax(), "event_id"],
)

In [ ]:
# --- KDE en CRS métrico, con un mínimo de peso (evita pesos exactamente 0
# que anularían por completo eventos reales de la muestra) ---
# --- KDE en CRS métrico, con un mínimo de peso (evita pesos exactamente 0
# que anularían por completo eventos reales de la muestra) ---
# Reproject events to the metric CRS
he_m = he.to_crs(CRS_METRIC)
# X coordinates in metres
xs = he_m.geometry.x.values
# Y coordinates in metres
ys = he_m.geometry.y.values
# Kernel weights from the event severity, with a small constant so no event has zero weight
pesos_kde = he["event_severity_score"].values + 0.01  # piso mínimo, ver nota arriba

# Fit a severity-weighted Gaussian kernel density
kde = gaussian_kde(np.vstack([xs, ys]), weights=pesos_kde, bw_method=KDE_BW)
# Log the kernel bandwidth
logger.info("KDE ajustado: bandwidth factor (Silverman)=%.1f m", kde.factor * np.std(np.hstack([xs, ys])))

# --- Grilla de evaluación: bounding box de Ecuador continental (excluye
# Galápagos, muy lejos, generaría una grilla enorme casi vacía) + margen ---
# --- Grilla de evaluación: bounding box de Ecuador continental (excluye
# Galápagos, muy lejos, generaría una grilla enorme casi vacía) + margen ---
# Name of the province column
provincia_col = "province" if "province" in he_m.columns else "province_name_official"
# Exclude the Galapagos Islands from the continental grid
continental = he_m[~he_m[provincia_col].str.upper().str.contains("GALAPAGOS|GALÁPAGOS", na=False)]
# Bounding box of continental events
minx, miny, maxx, maxy = continental.total_bounds
# Margin around the bounding box, in metres
margen = 20_000  # 20 km de margen
# Grid step in metres
paso = KDE_GRID_KM * 1_000
# Regular grid of x and y coordinates
xx, yy = np.meshgrid(
    np.arange(minx - margen, maxx + margen, paso),
    np.arange(miny - margen, maxy + margen, paso),
)
# Flatten the grid into a 2 x N array of points
grid_xy = np.vstack([xx.ravel(), yy.ravel()])
# Evaluate the density at every grid point
densidad = kde(grid_xy)
# Normalize the density to the 0-1 range
densidad_norm = (densidad - densidad.min()) / (densidad.max() - densidad.min())

# Build the density grid as a GeoDataFrame
grid_gdf = gpd.GeoDataFrame(
    {"densidad_kde": densidad, "densidad_kde_norm": densidad_norm},
    geometry=gpd.points_from_xy(grid_xy[0], grid_xy[1]),
    crs=CRS_METRIC,
).to_crs(4326)
# Log the number of grid points
logger.info("Grilla KDE evaluada: %d puntos (paso=%d km)", len(grid_gdf), KDE_GRID_KM)

**Nota de alcance:** la grilla KDE cubre Ecuador continental; Galápagos se
excluye de esta superficie continua (por su distancia extrema, incluirla
generaría una grilla dominada por océano vacío) pero sus eventos sí
están incluidos en el ajuste del propio KDE de forma marginal — su efecto
en la superficie continental es, por la distancia, numéricamente
despreciable. Los cantones de Galápagos sí se cubren en las Secciones
3-4 (Gi*/Local Moran's I) y en la agregación hexagonal (Sección 7).

**English:** Scope note: the KDE grid covers mainland Ecuador. Galápagos is excluded from this continuous surface (including it would produce a grid dominated by empty ocean), although its events are marginally included in the KDE fit; their effect on the mainland surface is numerically negligible. Galápagos cantons are covered in Sections 3–4 (Gi*/Local Moran's I) and in the hexagonal aggregation (Section 7).

## 7. Agregación hexagonal (H3 resolución 5, ~10 km)

En vez de rellenar todo el mapa de Ecuador con hexágonos vacíos (miles de
celdas sobre el océano u zonas sin eventos), se asigna cada evento a su
celda H3 y se agrega solo sobre las celdas que tienen al menos un evento
— el enfoque estándar de "hexbin" para datos puntuales, más eficiente y
sin celdas vacías que no aportan información.

**English:** 7. Agregación hexagonal (H3 resolución 5, ~10 km)

En vez de rellenar todo el mapa de Ecuador con hexágonos vacíos (miles de
celdas sobre el océano u zonas sin eventos), se asigna cada evento a su
celda H3 y se agrega solo sobre las celdas que tienen al menos un evento
— el enfoque estándar de "hexbin" para datos puntuales, más eficiente y
sin celdas vacías que no aportan información.

**English:** 7. Hexagonal aggregation (H3 resolution 5, about 10 km)

In [ ]:
# Assign each event to its H3 hexagon
he["h3_cell"] = he.apply(lambda r: h3.latlng_to_cell(r["latitude"], r["longitude"], H3_RES), axis=1)

# Aggregate events per hexagon (count and severity statistics)
hex_agg = he.groupby("h3_cell").agg(
    n_eventos=("event_id", "count"),
    severity_mean=("event_severity_score", "mean"),
    severity_max=("event_severity_score", "max"),
    severity_sum=("event_severity_score", "sum"),
    hazard_type_dominante=("hazard_type", lambda s: s.value_counts().idxmax()),
).reset_index()

# Shapely geometry helpers
import shapely.geometry as shp_geom
# Build each hexagon's polygon from its H3 cell
hex_agg["geometry"] = hex_agg["h3_cell"].apply(
    lambda c: shp_geom.Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)])
)
# Create the hexagon GeoDataFrame
hex_gdf = gpd.GeoDataFrame(hex_agg, geometry="geometry", crs=4326)

# Log the coverage check
logger.info(
    "Agregación hexagonal (res=%d, ~10km): %d celdas con >=1 evento, máximo %d eventos en una celda",
    H3_RES, len(hex_gdf), hex_gdf["n_eventos"].max(),
)
# Show the ten hexagons with the most events
hex_gdf.sort_values("n_eventos", ascending=False).head(10)[["h3_cell", "n_eventos", "severity_mean", "hazard_type_dominante"]]

## 8. Validación final y exportación

**English:** 8. Validación final y exportación

**English:** 8. Final validation and export

In [ ]:
# Validaciones antes de exportar
# Check hexagon counts add up to the number of events
assert hex_gdf["n_eventos"].sum() == len(he), "la suma de eventos por hexágono no coincide con el total de eventos"
# Check the density grid has no missing values
assert grid_gdf["densidad_kde"].notna().all(), "hay valores nulos en la grilla KDE"
# Check all hexagon geometries are valid
assert (hex_gdf.geometry.is_valid).all(), "hay geometrías de hexágono inválidas"

# Create the output folder if needed
out_dir = ensure_dir(processed_dir)

gi_df.to_parquet(out_dir / "hotspots_gi_star_canton_scour.parquet", index=False)
gi_df.to_csv(out_dir / "hotspots_gi_star_canton_scour.csv", index=False)

moran_df.to_parquet(out_dir / "local_moran_canton_scour.parquet", index=False)
moran_df.to_csv(out_dir / "local_moran_canton_scour.csv", index=False)

grid_gdf.to_file(out_dir / "kde_severidad_grid_scour.geojson", driver="GeoJSON")
# Export the density grid as CSV with coordinates
grid_gdf.drop(columns="geometry").assign(
    longitude=grid_gdf.geometry.x, latitude=grid_gdf.geometry.y
).to_csv(out_dir / "kde_severidad_grid_scour.csv", index=False)

hex_gdf.to_file(out_dir / "hexagonos_h3_res5_scour.geojson", driver="GeoJSON")
hex_gdf.drop(columns="geometry").to_csv(out_dir / "hexagonos_h3_res5_scour.csv", index=False)

logger.info("Fase 9 (scour) — todas las salidas exportadas a %s", out_dir)
print("Exportado: hotspots_gi_star_canton_scour.{parquet,csv}")
print("Exportado: local_moran_canton_scour.{parquet,csv}")
print("Exportado: kde_severidad_grid_scour.{geojson,csv}")
print("Exportado: hexagonos_h3_res5_scour.{geojson,csv}")

## 9. Resumen de hallazgos (ver también `docs/{methodology,executive}/09_spatial_analysis.md`)

Ejecutar las secciones anteriores y revisar los `logger.info`/`print` de
cada una para los números reales antes de redactar los documentos de
fase — esta celda es un recordatorio de qué preguntas debe responder el
resumen, no un resultado fijo:

- ¿Cuántos cantones son hotspot/coldspot significativo en `hci`, y
  coinciden con el top-15 de la Fase 8 §7?
- ¿Los hotspots de HCI están dominados por Severidad, Recurrencia,
  Persistencia o Magnitud ENSO (comparar las 5 capas de Gi*/Local
  Moran's I entre sí)?
- ¿La superficie KDE resalta alguna zona que no sea evidente en el mapa
  por cantón (p. ej. un corredor que cruza varios cantones pequeños)?
- ¿Qué hexágonos concentran más eventos/severidad, y a qué cantón(es)
  pertenecen?

**English:** 9. Resumen de hallazgos (ver también `docs/{methodology,executive}/09_spatial_analysis.md`)

Ejecutar las secciones anteriores y revisar los `logger.info`/`print` de
cada una para los números reales antes de redactar los documentos de
fase — esta celda es un recordatorio de qué preguntas debe responder el
resumen, no un resultado fijo:

- ¿Cuántos cantones son hotspot/coldspot significativo en `hci`, y
  coinciden con el top-15 de la Fase 8 §7?
- ¿Los hotspots de HCI están dominados por Severidad, Recurrencia,
  Persistencia o Magnitud ENSO (comparar las 5 capas de Gi*/Local
  Moran's I entre sí)?
- ¿La superficie KDE resalta alguna zona que no sea evidente en el mapa
  por cantón (p. ej. un corredor que cruza varios cantones pequeños)?
- ¿Qué hexágonos concentran más eventos/severidad, y a qué cantón(es)
  pertenecen?